# Child Education Risk Intelligence Platform

Notebook workflow aligned with the project presentation:

1. Data understanding and ingestion
2. Data validation and cleaning
3. Exploratory data analysis
4. Feature engineering
5. AI model development (XGBoost)
6. Model evaluation and explainability (SHAP)
7. Risk prioritization and intervention engine
8. BI layer outputs
9. API / dashboard integration
10. Prototype delivery

Place your CSV under `datasets/raw/` (for example `student_education_risk.csv`).

In [ ]:
from pathlib import Path

import pandas as pd

from datasphere.data.loader import discover_datasets, load_primary_dataset
from datasphere.ml.labels import construct_risk_label
from datasphere.ml.train import prepare_labeled_frame, train_model

DATASETS = discover_datasets()
DATASETS

In [ ]:
df = load_primary_dataset()
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
df.head()

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing.head(15)

In [ ]:
labeled = prepare_labeled_frame(df)
labeled["risk_level"].value_counts()

In [ ]:
numeric = labeled.select_dtypes(include="number")
if not numeric.empty:
    numeric.corr(numeric.get("attendance", numeric.columns[0]), method="pearson").sort_values(ascending=False).head(10)

In [ ]:
training_result = train_model(df)
training_result["classification_report"], training_result["confusion_matrix"]

In [ ]:
import joblib
import shap

pipeline = joblib.load(training_result["model_path"])
preprocessor = pipeline.named_steps["preprocessor"]
classifier = pipeline.named_steps["classifier"]

features = labeled.drop(columns=["risk_level"], errors="ignore")
sample = features.sample(min(500, len(features)), random_state=42)
transformed = preprocessor.transform(sample)
explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(transformed)
print("SHAP values computed for sample predictions.")

## Next steps

- Push the worked dataset to `datasets/raw/` on GitHub
- Re-run this notebook end to end
- Serve predictions through `POST /api/model/train` and the React dashboard